In [ ]:
import numpy as np
from Bio import Phylo
import sys
sys.path.append('../../pysimARG')
from clonal_genealogy import ClonalTree
from ClonalOrigin_ARG import ARG
from add_mutation import add_mutation
from homoplasy_index import homoplasy_index
from newick_to_tree import newick_to_tree
from G4_test import G4_test
from LD_r import LD_r
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim

In [2]:
np.random.seed(100)
clonal_tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

rho_site = 0.02
theta_site = 0.05
L = 2000
delta = 300

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

## ClonalOrigin seq

In [17]:
ARG_sim = ARG(clonal_tree, rho_site, L, delta, L, "seq")

In [18]:
ARG_sim.node_mat

array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]], shape=(304, 2000))

In [19]:
node_site = add_mutation(ARG_sim, theta_site)
node_site

array([[False,  True, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False],
       ...,
       [ True,  True, False, ..., False,  True, False],
       [ True,  True, False, ..., False,  True, False],
       [ True,  True,  True, ..., False,  True, False]], shape=(304, 2000))

In [20]:
mat = node_site[:10, :]
mat

array([[False,  True, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False],
       ...,
       [False,  True, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False]], shape=(10, 2000))

In [21]:
s_vec = np.full(19, np.nan)
s_vec

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan])

In [22]:
has_true = mat.any(axis=0)
has_false = ~mat.all(axis=0)
idx_seg = np.where(has_true & has_false)[0]

In [23]:
seg_near, seg_far = 0, 0
seg_20_50, seg_50_80 = 0, 0

In [24]:
ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
if idx_seg.size >= 2:
    for i in range(idx_seg.size - 1):
        for j in range(i + 1, idx_seg.size):
            dist_ij = idx_seg[j] - idx_seg[i]
            idx_pair = [idx_seg[i], idx_seg[j]]
            if dist_ij < L/2:
                ld_near += LD_r(mat[:, idx_pair])
                g4_near += G4_test(mat[:, idx_pair])
                seg_near += 1
            else:
                ld_far += LD_r(mat[:, idx_pair])
                g4_far += G4_test(mat[:, idx_pair])
                seg_far += 1
            if 20 <= dist_ij < 50:
                ld_20_50 += LD_r(mat[:, idx_pair])
                g4_20_50 += G4_test(mat[:, idx_pair])
                seg_20_50 += 1
            if 50 <= dist_ij <= 80:
                ld_50_80 += LD_r(mat[:, idx_pair])
                g4_50_80 += G4_test(mat[:, idx_pair])
                seg_50_80 += 1
    
    s_vec[0] = ld_near
    s_vec[1] = ld_far
    s_vec[2] = g4_near
    s_vec[3] = g4_far

    s_vec[4] = ld_near / seg_near if seg_near > 0 else 0
    s_vec[5] = ld_far /seg_far if seg_far > 0 else 0
    s_vec[6] = g4_near / seg_near if seg_near > 0 else 0
    s_vec[7] = g4_far / seg_far if seg_far > 0 else 0

    s_vec[8] = ld_20_50
    s_vec[9] = ld_50_80
    s_vec[10] = g4_20_50
    s_vec[11] = g4_50_80

    s_vec[12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
    s_vec[13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    s_vec[14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
    s_vec[15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
else:
    s_vec[:16] = 0

In [25]:
s_vec[:8]

array([3.33312528e+03, 7.35737247e+02, 1.14000000e+03, 3.50000000e+02,
       1.48899946e-01, 1.21488977e-01, 5.09269600e-02, 5.77939234e-02])

In [26]:
s_vec[8:16]

array([1.72731328e+02, 1.56924521e+02, 1.10000000e+01, 1.40000000e+01,
       1.98770228e-01, 1.83537451e-01, 1.26582278e-02, 1.63742690e-02])

In [27]:
seg_near, seg_far, seg_20_50, seg_50_80

(22385, 6056, 869, 855)

In [28]:
print(seg_near + seg_far)
print(idx_seg.size * (idx_seg.size - 1) / 2)

28441
28441.0


In [29]:
s_vec[16] = homoplasy_index(clonal_tree, node_site)

count_S = idx_seg.size
s_vec[17] = count_S / L

s_vec[18] = L

s_vec

array([3.33312528e+03, 7.35737247e+02, 1.14000000e+03, 3.50000000e+02,
       1.48899946e-01, 1.21488977e-01, 5.09269600e-02, 5.77939234e-02,
       1.72731328e+02, 1.56924521e+02, 1.10000000e+01, 1.40000000e+01,
       1.98770228e-01, 1.83537451e-01, 1.26582278e-02, 1.63742690e-02,
       3.09248555e-01, 1.19500000e-01, 2.00000000e+03])